First we check the GPU version available in the environment and install specific dependencies that are compatible with the detected GPU to prevent version conflicts.

In [1]:
# %%capture
import torch
major_version, minor_version = torch.cuda.get_device_capability()
print(major_version, minor_version)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
if major_version >= 8:
    !pip install --no-deps packaging ninja einops flash-attn xformers trl peft accelerate bitsandbytes
else:
    !pip install --no-deps xformers trl peft accelerate bitsandbytes
pass


8 6
Defaulting to user installation because normal site-packages is not writeable
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-61_l4b24/unsloth_01f21f8e03da42088951f5715ad1d546
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-61_l4b24/unsloth_01f21f8e03da42088951f5715ad1d546
  Resolved https://github.com/unslothai/unsloth.git to commit 933d9fe2cb2459f949ee2250e90a5b610d277eab
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Defaulting to user installation because normal site-packages is not writeable


load foundational model

In [1]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 8192 # Choose any! Llama 3 is up to 8k
dtype = None
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

fourbit_models = [
    "unsloth/mistral-7b-bnb-4bit",
    "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
    "unsloth/llama-2-7b-bnb-4bit",
    "unsloth/gemma-7b-bnb-4bit",
    "unsloth/gemma-7b-it-bnb-4bit",
    "unsloth/gemma-2b-bnb-4bit",
    "unsloth/gemma-2b-it-bnb-4bit",
    "unsloth/llama-3-8b-bnb-4bit",
]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit", # Llama-3 70b also works (just change the model name)
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)


/home/instruct/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth: Fast Llama patching release 2024.6
   \\   /|    GPU: NVIDIA A10G. Max memory: 22.095 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.3.1+cu121. CUDA = 8.6. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. Xformers = 0.0.26.post1. FA = True.
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


load finetuned model

In [10]:
max_seq_length = 8192  # Choose any! Llama 3 is up to 8k
dtype = None
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "model_ncs_full_tests_822", # YOUR MODEL YOU USED FOR TRAINING
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


['<|begin_of_text|>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nList the prime numbers contained within the range.\n\n### Input:\n1-50\n\n### Response:\n2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47<|end_of_text|>']

load checkpoint

In [24]:
from unsloth import FastLanguageModel
import torch
import os

max_seq_length = 8192  # Choose any! Llama 3 is up to 8k
dtype = None
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

# Check if there's a checkpoint directory
checkpoint_dir = "/home/instruct/outputs/checkpoint-2900"

if os.path.exists(checkpoint_dir):
    model, tokenizer = FastLanguageModel.from_pretrained(checkpoint_dir)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<|begin_of_text|>Below is an instruction that describes a task, paired with steps that provides the steps to make the test. Write a response that appropriately completes the test.

### instruction:
Write me a Robot Test Framework code test for NCS (kubernetes platform)

### steps:
tc_test_master_ls: SHH into one of the master nodes and than run ls command

### test:




# new test case

tc_test_master_ls
ssh_cmd="ssh -q -tt -o StrictHostKeyChecking=no ${S_SSH_TEST_USER}@${S_CENTRAL_MANAGER_IP}"
command="hostname"
${conn}= tcp.socket.open_to=${S_CENTRAL_MANAGER_IP},port=22
${std_out} = tcp.socket.send_command_on_connection=${conn},command=${command}
Log To Console "${std_out}"
tcp.socket.close=${conn}
<|end_of_text|>


inference

In [ ]:
# alpaca_prompt = Copied from above
EOS_TOKEN = tokenizer.eos_token
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
    [
        """Write a test case in Robot Framework for the NCS project that scale out a node.
        <prompt-end>
       """
    ], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 8192)